# This is a rough draft of my general workflow for Mini-Project 1

## Step 1: Load in the data and inspect it 

# From the data source we have a sort description of each dataset: 
- data.csv: "All of the text written/generated from the various different sources"
    - this is the main corpus: what the whole project will be built on 
- distribution.csv: "The distribution of data across all the different sources"
    - this will be useful for the preliminary corpus statistics section 
        - will tell us how many texts came from each source (i.e., which LLMs are represented int he data, how much of the data is human, etc.) without having to load all 800k rows 
- prompts.csv : "All the prompts/topics used for the writing/generation of all the text in data.csv/data.parquet"
    - could be interesting for context of the text in data.csv but not central to the analysis 
        - might be nice to look at just to know what prompts were used (help with some choices I make or limitations I discuss)

In [8]:
import pandas as pd
import os 

# Build path relative to this notebook's location
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
data_dir = os.path.join(notebook_dir, '..', 'data')

# read in just the first 3 rows of each dataset to take a look
for fname in ['data.csv', 'distribution.csv', 'prompts.csv']:
    df = pd.read_csv(os.path.join(data_dir, fname), nrows=3)
    print(f"\n=== {fname} ===")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(3))


=== data.csv ===
Columns: ['text', 'source', 'prompt_id', 'text_length', 'word_count']
                                                text    source  prompt_id  \
0  Federal law supersedes state law, and cannabis...  Bloom-7B          0   
1  Miles feels restless after working all day. He...  Bloom-7B          0   
2  So first of I am danish. That means that I fol...  Bloom-7B          0   

   text_length  word_count  
0          967         157  
1         5068         778  
2         1602         267  

=== distribution.csv ===
Columns: ['Source', 'Number of Samples', 'Percentage of Total Data', 'Text Length Sum', 'Text Length Mean', 'Text Length Median', 'Text Length Std', 'Text Length Max', 'Text Length Min', 'Word Count Sum', 'Word Count Mean', 'Word Count Median', 'Word Count Std', 'Word Count Max', 'Word Count Min']
             Source  Number of Samples Percentage of Total Data  \
0             Human             347692                 44.0718%   
1           GPT-3.5         

## From this output we've learned: 
- data.csv has the columns we'll need: 
    - text: is the corpus
    - source: the label (not just Human/LLM binary, but the actual model name like Bloom-7B, GPT-3.5, etc.)
    - prompt_id links to the prompts file, and text_length/word_count are pre-computed which is nice for corpus statistics.
- distribution.csv provides various summary statistics 
- prompts.csv tells us the writing prompts
    - prompt 0 is "Undefined" (some texts had no specific prompt)

### From this my two ideas for classifications methods are either
- Option A: Binary classification human (0) vs. All LLMs combined (1)
- Option B: Multi-classification: human vs (the different AI models: vs. GPT-3.5 vs. Bloom-7B vs. Text-Davinci-003 vs. ...)

### To better inform my decision on a classification method, I will explore the number of unique datasources 


In [14]:
# Load full distribution to see all sources
dist_df = pd.read_csv(os.path.join(data_dir, 'distribution.csv'))
print(dist_df[['Source', 'Number of Samples', 'Percentage of Total Data']])

                      Source  Number of Samples Percentage of Total Data
0                      Human             347692                 44.0718%
1                    GPT-3.5              52346                  6.6351%
2           Text-Davinci-003              22860                  2.8976%
3           Text-Davinci-002              21436                  2.7171%
4                   OPT-1.3B              18467                  2.3408%
..                       ...                ...                      ...
58                Toppy-M-7B                433                  0.0549%
59                LLaMA-2-7B                409                  0.0518%
60      Dolphin-Mixtral-8x7B                407                  0.0516%
61            Cohere-Command                390                  0.0494%
62  Dolphin-2.5-Mixtral-8x7B                228                  0.0289%

[63 rows x 3 columns]


### Based on my general purpose for this analysis I will do binary classification, as I think the general human vs. AI difference is more interesting/better aligns with the goal of my study 
### Overall: the more interesting question isn't if we can tell a difference between LLMs... its the general question: ("can we tell human writing from machine writing at all") = more culturally meaningful
- Further supporting this decision is the output above: 
    - There are 63 sources total (62 LLMs + humans), and many of the LLMs have tiny sample sizes so if I tried to do multi-class classification between all the different LLMs - most would end up so small that the later model wouldn't learn anything meaningful from them  
- Human vs. AI is more balanced (~347K human texts. vs ~441K AI texts)

# <span style="color: red;">FOR LATER WRITEUP- Limitation</span>
- The AI side (of the binary classification) is actually an aggregation of very different models (GPT family, Bloom, etc.) which vary a lot in quality and style. 
    - Worth flagging as a limitation — "AI writing" isn't monolithic, and collapsing 62 models into one label introduces heterogeneity that could affect both classification and topic modeling results.

# 1st Main Step: Load and Sample the Data 
- Because data.csv is 800k rows (that would take a lot of RAM), I will load in a reproducible random sample (~10,000 rows)
- Will also create the binary label in this step where (Human=0, all others(LLM's) = 1)

In [16]:
import numpy as np

# configuration step
    # defining key parameters for sampling 
SAMPLE_SIZE = 10000
RANDOM_SEED = 230   # setting a random seed for reproducibility (can be anything - INFO 230 - for fun!)

# using os path for reproducibility on another machine 
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
data_dir = os.path.join(notebook_dir, '..', 'data')

# First loading in full dataset then immediately taking a random sample to keep computation manageable
print("Loading data.csv...")
ai_human_df = pd.read_csv(os.path.join(data_dir, 'data.csv'))
print(f"Full dataset size: {len(ai_human_df):,} rows")

# Reproducible random sample
ai_human_df_sample = ai_human_df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Sample size: {len(ai_human_df_sample):,} rows")

# Create the binary label 
    # original data had 62+ unique LLM names plus 'human' 
    # binary classification simplify this to: 
        # Human = 0, all LLMs = 1

ai_human_df_sample['label'] = (ai_human_df_sample['source'] != 'Human').astype(int)

print(f"\nLabel distribution in sample:")
print(ai_human_df_sample['label'].value_counts().rename({0: 'Human', 1: 'AI'}))
print(f"\nSample preview:")
ai_human_df_sample[['text', 'source', 'label', 'word_count']].head(3)

Loading data.csv...
Full dataset size: 788,922 rows
Sample size: 10,000 rows

Label distribution in sample:
label
AI       5548
Human    4452
Name: count, dtype: int64

Sample preview:


,text,source,label,word_count
0,Oxygen gas (O 2) can be toxic at elevated part...,Human,0,105
1,The Tortoise and the Rabbit: A Story of Determ...,Mistral-7B,1,401
2,"When we write words from another language, lik...",Text-Davinci-003,1,68


# Step 2: Preliminary corpus Statistics

# Step 3: Chunking 